# 01 - Exploracion de una escena Sentinel-2 L2A

Hito de Semana 2: cargar la escena, enmascararla y mostrar la firma de un
pixel superpuesta a la de referencia, con las 12 bandas alineadas.

El notebook no calcula nada por su cuenta: llama al paquete `mineralmap` y
comprueba que lo que devuelve cumple los contratos de
`docs/es/decisiones_tecnicas.md`. Todo lo que aqui se verifica esta ademas
cubierto por `tests/`; el notebook existe para poder *ver* el resultado.

## 1. Setup

`autoreload` para que editar el paquete se refleje sin reiniciar el kernel.
El `chdir` deja el directorio de trabajo en la raiz del repositorio: las rutas
del proyecto (`data/raw/`, `outputs/figures/`) son relativas a la raiz, y un
notebook se puede lanzar tanto desde ahi como desde `notebooks/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

RAIZ = Path.cwd()
print("Directorio de trabajo:", RAIZ)

import matplotlib.pyplot as plt
import numpy as np
from rasterio.windows import Window

from mineralmap.algorithms.sam import SAM
from mineralmap.config import BAND_ORDER, DEFAULT_PLATFORM, SAM_BANDS, band_wavelengths
from mineralmap.preprocessing.masking import apply_mask
from mineralmap.preprocessing.scene_builder import build_scene_from_safe
from mineralmap.spectral.endmembers import get_reference_spectrum
from mineralmap.visualization.maps import rgb_composite
from mineralmap.visualization.spectra import plot_spectra

## 2. Cargar la escena

Ventana de 256x256 px a proposito, para que el notebook corra en segundos.

**La ventana autoritativa del proyecto es 2000x2000 px** (`col_off=2700`,
`row_off=650`), o sea 40x40 km a 20 m, y es la que define la zona de estudio
en `configs/cerro_colorado_kaolinite.yaml`. Esta de 256x256 es una submuestra del
mismo origen para explorar; comparte esquina superior izquierda con la
autoritativa, asi que lo que se ve aqui es una esquina de aquella.

In [ ]:
AOI_EXPLORACION = Window(col_off=2700, row_off=650, width=256, height=256)

scene = build_scene_from_safe(aoi=AOI_EXPLORACION, verbose=True)

## 3. Contrato de la escena

Antes de mirar ningun numero, comprobar que el objeto es el que dice el
contrato de `Scene`: cubo `(n_bandas, alto, ancho)` en `float32`, bandas en el
orden de `BAND_ORDER`, y `mask` con `True` = pixel valido.

In [ ]:
print("cube.shape      :", scene.cube.shape)
print("cube.dtype      :", scene.cube.dtype)
print("band_names      :", scene.band_names)
print("crs (EPSG)      :", scene.crs.to_epsg())
print("aoi_window      :", scene.meta["aoi_window"])
print()

print("Resolucion nativa por banda (meta['bands_source']):")
for banda, resolucion in scene.meta["bands_source"].items():
    print(f"  {banda:<4} -> R{resolucion}m")
print()

resumen_scl = dict(scene.meta["scl_summary"])
pct_meta = resumen_scl.pop("pct_validos")
print("Composicion SCL del AOI (meta['scl_summary']):")
for clase, porcentaje in sorted(resumen_scl.items(), key=lambda p: -p[1]):
    print(f"  {clase:<22} {porcentaje:8.4f} %")
print()

pct_validos = 100.0 * scene.mask.mean()
print(f"% validos (mask)     : {pct_validos:.4f} %  ({int(scene.mask.sum()):,} px)")
print(f"% validos (scl_summary): {pct_meta:.4f} %")

## 4. Enmascarar

`build_scene_from_safe` entrega el cubo **sin** enmascarar y la validez aparte,
en `scene.mask`: quien decide aplicarla es el consumidor, porque un pixel que
paso a `NaN` no se recupera sin releer el producto. `apply_mask` es ese paso
explicito, y devuelve una copia.

En este AOI la mascara no deberia quitar nada: es desierto despejado.

In [ ]:
cubo_enmascarado = apply_mask(scene.cube, scene.mask)

nan_antes = int(np.isnan(scene.cube).sum())
nan_despues = int(np.isnan(cubo_enmascarado).sum())
px_totales = scene.mask.size

print(f"Pixeles del AOI        : {px_totales:,}")
print(f"Pixeles invalidos      : {int((~scene.mask).sum()):,}")
print(f"NaN en el cubo, antes  : {nan_antes:,}")
print(f"NaN en el cubo, despues: {nan_despues:,}")
print(f"NaN agregados          : {nan_despues - nan_antes:,}"
      f"  (= {int((~scene.mask).sum()):,} px x {scene.cube.shape[0]} bandas)")
print()
print("El cubo original no se toca (apply_mask devuelve una copia):",
      not np.isnan(scene.cube).any() or nan_antes > 0)

## 5. RGB del AOI

Para ubicarse geograficamente. B4/B3/B2 como R/G/B, con realce por percentiles
2-98 banda a banda. El realce vive en `visualization/maps.py` y es el mismo
que usa `scripts/visualizar_rgb.py`: dos copias de esa logica pueden divergir
sin que ninguna de las dos figuras se vea mal.

In [ ]:
rgb = rgb_composite(scene)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb)
ax.set_title(
    f"AOI de exploracion 256x256 px a 20 m ({scene.meta['tile_id']})\n"
    f"{scene.meta['sensing_date'][:10]}  -  RGB B4/B3/B2, realce P2-P98"
)
ax.axis("off")
plt.show()

## 6. Verificacion de alineamiento (entregable de Track B)

Este es el punto del notebook. Todo el proyecto asume que `cube[i]`,
`band_names[i]` y `firma[i]` hablan de la misma banda, y ninguna de las tres
cosas lo comprueba por su cuenta: si estuvieran cruzadas, el SAM seguiria
devolviendo un angulo perfectamente calculado sobre bandas equivocadas.

La tabla pone las cuatro columnas una al lado de la otra, y los `assert`
posteriores hacen que el notebook falle ruidosamente si algo no cuadra.

In [ ]:
lambdas = band_wavelengths(BAND_ORDER, platform=DEFAULT_PLATFORM)
ref_usgs = get_reference_spectrum("kaolinite", band_order=BAND_ORDER)

# Pixel central del AOI, como muestra para la tabla.
fila_centro, col_centro = np.array(scene.cube.shape[1:]) // 2
pixel_central = cubo_enmascarado[:, fila_centro, col_centro]

cabecera = f"{'idx':>3}  {'banda':<5} {'lambda_' + DEFAULT_PLATFORM + ' (nm)':>16}"
cabecera += f" {'ref_USGS':>10} {'pixel':>10}"
print(cabecera)
print("-" * len(cabecera))
for i, banda in enumerate(scene.band_names):
    print(f"{i:>3}  {banda:<5} {lambdas[i]:>16.1f} "
          f"{ref_usgs[i]:>10.4f} {pixel_central[i]:>10.4f}")

In [ ]:
assert scene.band_names == BAND_ORDER, (
    f"El Scene no trae el contrato de bandas: {scene.band_names}"
)
assert len(ref_usgs) == scene.cube.shape[0], (
    f"La firma tiene {len(ref_usgs)} valores y el cubo {scene.cube.shape[0]} bandas"
)
assert len(lambdas) == scene.cube.shape[0]
assert lambdas == sorted(lambdas), "Las longitudes de onda no son crecientes"
assert "B10" not in scene.band_names, "B10 no existe en el producto L2A"

print("Alineamiento verificado:")
print(f"  {scene.cube.shape[0]} bandas del cubo == {len(ref_usgs)} valores de la firma")
print(f"  == {len(lambdas)} longitudes de onda, crecientes de "
      f"{lambdas[0]:.1f} a {lambdas[-1]:.1f} nm")

## 7. Firma del pixel superpuesta a la de referencia

**Criterio de eleccion del pixel, explicito.** Se grafican dos, no uno
cualquiera:

- **El de menor angulo SAM del AOI**: es el mejor candidato a caolinita que hay
  en esta ventana. El angulo se calcula sobre `SAM_BANDS` (9 bandas), que es el
  subconjunto que consumira el detector, no sobre las 12.
- **El pixel central**, como control. No fue elegido por parecerse a nada, asi
  que sirve para ver cuanto se parecen entre si dos pixeles cualesquiera de
  este desierto, y por lo tanto cuanto discrimina de verdad el mejor candidato.

Las firmas van normalizadas a norma L2, que es lo que ve el SAM: compara
direcciones, no magnitudes. Sin normalizar, la firma de laboratorio (AREF) y el
pixel real (reflectancia superficial) quedan separados por un factor que se
imprime abajo, y el grafico sugiere un desalineamiento que no existe.

In [ ]:
indices_sam = [BAND_ORDER.index(banda) for banda in SAM_BANDS]
ref_sam = get_reference_spectrum("kaolinite", band_order=SAM_BANDS)

mapa_sam9 = SAM().predict(cubo_enmascarado[indices_sam], ref_sam)
mapa_sam12 = SAM().predict(cubo_enmascarado, ref_usgs)

fila_mejor, col_mejor = np.unravel_index(np.nanargmin(mapa_sam9), mapa_sam9.shape)
pixel_mejor = cubo_enmascarado[:, fila_mejor, col_mejor]

print(f"Mejor pixel (menor angulo SAM sobre {len(SAM_BANDS)} bandas): "
      f"fila {fila_mejor}, col {col_mejor}")
print(f"  angulo 9 bandas : {mapa_sam9[fila_mejor, col_mejor]:.4f} rad")
print(f"  angulo 12 bandas: {mapa_sam12[fila_mejor, col_mejor]:.4f} rad")
print(f"Pixel central (control): fila {fila_centro}, col {col_centro}")
print(f"  angulo 9 bandas : {mapa_sam9[fila_centro, col_centro]:.4f} rad")
print()

print("Razon de magnitudes (por eso se normaliza el grafico):")
print(f"  media firma USGS / media pixel mejor = "
      f"{np.nanmean(ref_usgs) / np.nanmean(pixel_mejor):.2f}")
print()

# Distribucion del angulo sobre todo el AOI, con los dos conjuntos de bandas.
# Es el numero que decide si el umbral del config detecta algo o nada.
UMBRAL_CONFIG_RAD = 0.1  # configs/cerro_colorado_kaolinite.yaml
print(f"{'conjunto':<18} {'min':>8} {'mediana':>9} {'max':>8} {'bajo umbral':>12}")
print("-" * 58)
for nombre, mapa in (("SAM_BANDS (9)", mapa_sam9), ("BAND_ORDER (12)", mapa_sam12)):
    print(f"{nombre:<18} {np.nanmin(mapa):>8.4f} {np.nanmedian(mapa):>9.4f} "
          f"{np.nanmax(mapa):>8.4f} "
          f"{int(np.nansum(mapa <= UMBRAL_CONFIG_RAD)):>12,}")
print()
print(f"Con angle_threshold_rad = {UMBRAL_CONFIG_RAD} el mapa de deteccion "
      f"quedaria vacio en este AOI.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

plot_spectra(
    {
        "Caolinita USGS splib07 (KGa-1)": ref_usgs,
        f"Mejor pixel SAM (f{fila_mejor}, c{col_mejor})": pixel_mejor,
        f"Pixel central, control (f{fila_centro}, c{col_centro})": pixel_central,
    },
    band_order=BAND_ORDER,
    normalize="l2",
    ax=ax,
    title=(
        "Firma de referencia vs. pixeles del AOI, 12 bandas alineadas\n"
        f"{scene.meta['tile_id']} - {scene.meta['sensing_date'][:10]} "
        f"({DEFAULT_PLATFORM})"
    ),
)

RUTA_FIGURA = RAIZ / "outputs" / "figures" / "kaolinite_signature_vs_pixel.png"
RUTA_FIGURA.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(RUTA_FIGURA, dpi=150, bbox_inches="tight")
print(f"Guardado: {RUTA_FIGURA.relative_to(RAIZ)}")
plt.show()

## 8. Que se ve, que no se puede concluir todavia

**Que se ve.** Las tres curvas estan alineadas banda a banda sobre el mismo eje
de longitud de onda: el entregable de Track B se cumple. El hueco entre 945 y
1610 nm no es un error del grafico, es que B10 no existe en el producto L2A y
ahi no hay ninguna medicion.

Y se ve algo mas, que es lo importante: **la referencia cae en picada de B11
(1610 nm) a B12 (2186 nm) y ninguno de los dos pixeles la acompana.** Esa caida
es la absorcion Al-OH del doblete 2160/2200 nm, o sea el unico rasgo del
espectro que distingue caolinita de un suelo cualquiera. Los dos pixeles
quedan casi planos en ese tramo. El "mejor pixel" del AOI lo es por parecerse a
la referencia en el visible y el infrarrojo cercano, que es donde ambas curvas
solo describen una superficie clara y seca, no por tener el rasgo diagnostico.

**Que no se puede concluir.** Que el mejor pixel sea caolinita; el grafico dice
mas bien lo contrario. Es el minimo de esta ventana, que es cosa distinta: en un
AOI de 256x256 siempre hay un minimo, lo tenga o no. El pixel de control existe
justamente para medir eso, y la distancia entre ambos (0.170 vs 0.273 rad) es lo
unico que dice algo. Confirmarlo o descartarlo necesita el paso de validacion
geologica (`validation/geology.py`), que todavia no esta.

**El dato duro: con el umbral actual el mapa saldria vacio.**
`configs/cerro_colorado_kaolinite.yaml` fija `angle_threshold_rad: 0.1`, y el angulo
minimo medido en esta ventana es 0.170 rad. Ni un solo pixel de los 65 536 quedaria
bajo el umbral. **No se toca el config en este trabajo**: el 0.1 es un valor
heredado que hay que reemplazar con un criterio, no con otro numero elegido a
ojo, y ese criterio es trabajo del paso de calibracion. Vale la pena notar que
lo de arriba explica por que: si ningun pixel tiene el rasgo Al-OH, ningun
umbral razonable va a encontrar caolinita aqui, y bajarlo hasta que "detecte
algo" solo produciria falsos positivos.

**9 bandas vs 12.** Medido sobre este AOI:

| bandas | min | mediana | max |
|--------|-----|---------|-----|
| `SAM_BANDS` (9)  | 0.170 | 0.313 | 0.490 |
| `BAND_ORDER` (12) | 0.166 | 0.247 | 0.426 |

Las dos distribuciones son practicamente la misma, que es exactamente lo
esperado y no un argumento a favor de ninguna. Quitar B1, B8 y B9 no cambia el
resultado sobre desierto despejado porque ahi las tres aportan poca varianza
util. La razon para preferir 9 no es el rendimiento en esta escena, sino la que
esta documentada en `decisiones_tecnicas.md`: B1 y B9 miden atmosfera y B8 le
daria peso doble al infrarrojo cercano. Lo que este numero si aporta es que el
recorte **no cuesta nada**, que es lo que habia que comprobar antes de fijarlo.